In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {name} ({mem:.1f} GB)')

Mon Apr 27 11:30:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q unsloth sacrebleu datasets
print("✅ Cài xong!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 29.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.0 MB/s eta 0:00:00:00:01
  

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

secrets  = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)

SAVE_DIR = '/kaggle/working/llama3-en-vi'
os.makedirs(SAVE_DIR, exist_ok=True)

print('✅ Đăng nhập thành công!')
print(f'✅ Output lưu tại: {SAVE_DIR}')

✅ Đăng nhập thành công!
✅ Output lưu tại: /kaggle/working/llama3-en-vi


In [4]:
from datasets import load_dataset

print('📥 Đang load PhoMT...')
dataset = load_dataset('ura-hcmut/PhoMT')
print(dataset)

📥 Đang load PhoMT...


README.md:   0%|          | 0.00/653 [00:00<?, ?B/s]

PhoMT_training.csv:   0%|          | 0.00/525M [00:00<?, ?B/s]

PhoMT_validation.csv: 0.00B [00:00, ?B/s]

PhoMT_test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2977999 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/18720 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/19151 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['en', 'vi'],
        num_rows: 2977999
    })
    validation: Dataset({
        features: ['en', 'vi'],
        num_rows: 18720
    })
    test: Dataset({
        features: ['en', 'vi'],
        num_rows: 19151
    })
})


In [5]:
NUM_TRAIN = 100_000
NUM_VAL   = 2_000
NUM_TEST  = 500

SYSTEM_PROMPT = (
    'You are a professional English-Vietnamese translator. '
    'Translate the given English text to Vietnamese accurately and naturally.'
)

def filter_length(example):
    if not example['en'] or not example['vi']:
        return False
    en_len = len(example['en'].split())
    vi_len = len(example['vi'].split())
    return 3 <= en_len <= 80 and 3 <= vi_len <= 100

def format_instruction(example):
    return {
        'text': (
            '<|begin_of_text|>'
            '<|start_header_id|>system<|end_header_id|>\n'
            f'{SYSTEM_PROMPT}<|eot_id|>\n'
            '<|start_header_id|>user<|end_header_id|>\n'
            f'Translate to Vietnamese:\n{example["en"]}<|eot_id|>\n'
            '<|start_header_id|>assistant<|end_header_id|>\n'
            f'{example["vi"]}<|eot_id|>'
        )
    }

print('⚙️ Đang filter & format...')
train_dataset = (
    dataset['train']
    .filter(filter_length)
    .select(range(NUM_TRAIN))
    .map(format_instruction, remove_columns=['en', 'vi'])
)
val_dataset = (
    dataset['validation']
    .filter(filter_length)
    .select(range(NUM_VAL))
    .map(format_instruction, remove_columns=['en', 'vi'])
)

print(f'✅ Train : {len(train_dataset):,} samples')
print(f'✅ Val   : {len(val_dataset):,} samples')
print('\n📝 Ví dụ:')
print(train_dataset[0]['text'])

⚙️ Đang filter & format...


Filter:   0%|          | 0/2977999 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/18720 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Train : 100,000 samples
✅ Val   : 2,000 samples

📝 Ví dụ:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a professional English-Vietnamese translator. Translate the given English text to Vietnamese accurately and naturally.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Translate to Vietnamese:
It begins with a countdown.<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
Câu chuyện bắt đầu với buổi lễ đếm ngược.<|eot_id|>


In [6]:
from unsloth import FastLanguageModel, is_bf16_supported

print('📥 Đang load model... (~3-5 phút)')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct',  # Không cần token riêng
    max_seq_length=256,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

model.print_trainable_parameters()
print('✅ Model sẵn sàng!')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
📥 Đang load model... (~3-5 phút)
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.8 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511
✅ Model sẵn sàng!


In [7]:
import torch
from unsloth.trainer import UnslothTrainer, UnslothTrainingArguments

num_gpus = torch.cuda.device_count()
print(f'🖥️ Số GPU: {num_gpus}')

training_args = UnslothTrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=1,              # Giảm từ 3 xuống 1 epoch
    per_device_train_batch_size=16,  # Tăng từ 8 lên 16
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    bf16=is_bf16_supported(),
    fp16=not is_bf16_supported(),
    logging_steps=100,
    eval_steps=1000,                 # Ít eval hơn
    save_steps=1000,
    eval_strategy='steps',
    save_strategy='steps',
    warmup_steps=50,
    lr_scheduler_type='cosine',
    save_total_limit=1,
    load_best_model_at_end=True,
    report_to='none',
)

trainer = UnslothTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field='text',
    max_seq_length=256,
    tokenizer=tokenizer,
)

steps_per_epoch = len(train_dataset) // (
    training_args.per_device_train_batch_size * num_gpus
    * training_args.gradient_accumulation_steps
)
print(f'📊 Steps/epoch : {steps_per_epoch:,}')
print(f'📊 Tổng steps  : {steps_per_epoch * training_args.num_train_epochs:,}')
print('🚀 Bắt đầu training...')

trainer.train()
print('✅ Training xong!')

🖥️ Số GPU: 2


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/100000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
📊 Steps/epoch : 1,562
📊 Tổng steps  : 1,562
🚀 Bắt đầu training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 1 | Total steps = 3,125
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
1000,1.181690,1.192859
2000,1.163982,1.174120
3000,1.153734,1.166361
3125,1.140524,1.166331


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/llama3-en-vi/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/llama3-en-vi/checkpoint-2000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/llama3-en-vi/checkpoint-3000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/llama3-en-vi/checkpoint-3125/tokenizer_config.json.


✅ Training xong!


In [8]:
import os

ADAPTER_DIR = f'{SAVE_DIR}/adapter-final'
os.makedirs(ADAPTER_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
) / 1e6
print(f'✅ Adapter lưu tại: {ADAPTER_DIR}')
print(f'   Kích thước: {size:.1f} MB')
print('\n📥 Tải về: tab Output góc phải → llama3-en-vi → Download')

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/llama3-en-vi/adapter-final/tokenizer_config.json.


✅ Adapter lưu tại: /kaggle/working/llama3-en-vi/adapter-final
   Kích thước: 114.6 MB

📥 Tải về: tab Output góc phải → llama3-en-vi → Download


In [9]:
from sacrebleu.metrics import BLEU
from tqdm import tqdm

bleu_metric = BLEU()
model.eval()

def translate(text, max_new_tokens=150):
    prompt = (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n'
        f'{SYSTEM_PROMPT}<|eot_id|>\n'
        '<|start_header_id|>user<|end_header_id|>\n'
        f'Translate to Vietnamese:\n{text}<|eot_id|>\n'
        '<|start_header_id|>assistant<|end_header_id|>\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return decoded.strip()

test_samples = dataset['test'].filter(filter_length).select(range(NUM_TEST))
hypotheses, references = [], []

print(f'🔍 Evaluating {NUM_TEST} câu...')
for sample in tqdm(test_samples):
    hypotheses.append(translate(sample['en']))
    references.append([sample['vi']])

score = bleu_metric.corpus_score(hypotheses, references)
print(f'\n📊 BLEU Score: {score}')
print('   > 25: Tốt | > 30: Rất tốt | > 35: Xuất sắc') 

Filter:   0%|          | 0/19151 [00:00<?, ? examples/s]

🔍 Evaluating 500 câu...


  0%|          | 0/500 [00:00<?, ?it/s]Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, 


📊 BLEU Score: BLEU = 25.98 90.9/47.6/20.0/5.3 (BP = 1.000 ratio = 1.000 hyp_len = 22 ref_len = 22)
   > 25: Tốt | > 30: Rất tốt | > 35: Xuất sắc


In [10]:
test_sentences = [
    'Artificial intelligence is transforming the world.',
    'She walked slowly along the river bank, enjoying the sunset.',
    'The meeting has been postponed to next Monday.',
    'Vietnam is a beautiful country with a rich cultural heritage.',
]

print('🌐 Kết quả dịch:\n' + '─' * 60)
for sentence in test_sentences:
    print(f'EN: {sentence}')
    print(f'VI: {translate(sentence)}')
    print('─' * 60)

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🌐 Kết quả dịch:
────────────────────────────────────────────────────────────
EN: Artificial intelligence is transforming the world.


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


VI: Kỹ thuật trí tuệ nhân tạo đang thay đổi thế giới.
────────────────────────────────────────────────────────────
EN: She walked slowly along the river bank, enjoying the sunset.


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


VI: Cô ấy đi chậm rãi dọc bờ sông, tận hưởng ánh hoàng hôn.
────────────────────────────────────────────────────────────
EN: The meeting has been postponed to next Monday.


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


VI: Hội nghị đã bị hoãn đến thứ Hai tới.
────────────────────────────────────────────────────────────
EN: Vietnam is a beautiful country with a rich cultural heritage.
VI: Việt Nam là một đất nước đẹp với di sản văn hoá phong phú.
────────────────────────────────────────────────────────────


In [11]:
!pip install -q gradio

import gradio as gr
import torch

SYSTEM_PROMPT = (
    'You are a professional English-Vietnamese translator. '
    'Translate the given English text to Vietnamese accurately and naturally.'
)

def translate(text):
    if not text.strip():
        return "⚠️ Vui lòng nhập văn bản!"
    prompt = (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n'
        f'{SYSTEM_PROMPT}<|eot_id|>\n'
        '<|start_header_id|>user<|end_header_id|>\n'
        f'Translate to Vietnamese:\n{text}<|eot_id|>\n'
        '<|start_header_id|>assistant<|end_header_id|>\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

demo = gr.Interface(
    fn=translate,
    inputs=gr.Textbox(
        lines=5,
        placeholder="Nhập văn bản tiếng Anh...",
        label="🇺🇸 English"
    ),
    outputs=gr.Textbox(
        lines=5,
        label="🇻🇳 Tiếng Việt"
    ),
    title="🦙 LLaMA En→Vi Translator",
    description="Fine-tuned LLaMA 3.2 trên PhoMT 100k samples",
    examples=[
        ["Artificial intelligence is transforming the world."],
        ["She walked slowly along the river bank, enjoying the sunset."],
        ["The meeting has been postponed to next Monday."],
        ["Vietnam is a beautiful country with a rich cultural heritage."],
    ],
    theme=gr.themes.Soft(),
)

# share=True tự tạo public URL, dùng được ~72 giờ
demo.launch(share=True, quiet=True)

* Running on public URL: https://865188e97c792729fc.gradio.live


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=

In [12]:
import shutil

# Zip toàn bộ folder llama3-en-vi
shutil.make_archive(
    '/kaggle/working/llama3-en-vi-model',  # Tên file zip output
    'zip',                                  # Định dạng
    '/kaggle/working',                      # Thư mục gốc
    'llama3-en-vi'                          # Thư mục cần zip
)

print('✅ Đã tạo file: /kaggle/working/llama3-en-vi-model.zip')

✅ Đã tạo file: /kaggle/working/llama3-en-vi-model.zip
